# Hong Kong CHP — Influenza & COVID-19 Open Data

This notebook demonstrates the **Hong Kong CHP** accessor (`hk_chp`), which reads the
structured surveillance datasets that the Hong Kong Department of Health's Centre for
Health Protection (CHP) publishes through the [data.gov.hk](https://data.gov.hk) open data portal.

- **Flu Express figures data**: weekly influenza surveillance since **2014** — sentinel ILI
  consultation rates, laboratory detections/positivity by type and subtype, ILI outbreaks,
  public hospital admission rates and severe influenza cases
- **COVID-19 situation reports**: daily cumulative time series (2020 → 2023; upstream
  updates ceased on **19 March 2023**)

No scraping or PDF parsing is needed — everything comes from direct CSV endpoints.

## 1. Setup and discovery

In [1]:
from epidatasets.sources.hk_chp import HongKongCHPAccessor

hk = HongKongCHPAccessor()  # CSVs cached on disk, TTL 7 days

hk.list_countries()

,country_code,country_name
0,HK,Hong Kong SAR


## 2. Weekly influenza surveillance (Flu Express)

`get_influenza_data()` returns the full weekly table with normalized column names
(see the accessor's `FLU_COLUMN_MAP` and the official
[data dictionary](https://www.chp.gov.hk/files/pdf/flux_spec_en.pdf)).

In [2]:
flu = hk.get_influenza_data(year=2024)
print(f"{len(flu)} weeks available for 2024")
flu[["year", "week", "week_start", "week_end", "influenza_positivity",
     "ili_rate_private_practitioner_per_1000"]].head()

INFO:epidatasets.sources.hk_chp:Using cached CSV: /home/fccoelho/.cache/epidatasets/hk_chp/flux_data.csv
INFO:epidatasets.sources.hk_chp:Retrieved 52 weekly influenza records


52 weeks available for 2024


,year,week,week_start,week_end,influenza_positivity,ili_rate_private_practitioner_per_1000
0,2024,1,2023-12-31,2024-01-06,0.1177,48.3
1,2024,2,2024-01-07,2024-01-13,0.1067,61.7
2,2024,3,2024-01-14,2024-01-20,0.0691,55.8
3,2024,4,2024-01-21,2024-01-27,0.0490,33.8
4,2024,5,2024-01-28,2024-02-03,0.0522,44.4


### Sentinel ILI consultation rates

In [ ]:
ili = hk.get_ili_rates(year=2024)
ili.plot(x="week_start", y=["ili_rate_family_medicine_per_1000",
                            "ili_rate_private_practitioner_per_1000"],
         figsize=(11, 4), title="ILI consultations per 1,000 — Hong Kong, 2024");

### Laboratory surveillance: detections and positivity

In [ ]:
pos = hk.get_influenza_positivity(year=2024)
ax = pos.plot(x="week_start", y=["influenza_a_h1_detections", "influenza_a_h3_detections",
                                 "influenza_b_detections"],
              figsize=(11, 4), title="Influenza detections by type — Hong Kong, 2024")
pos.plot(x="week_start", y="influenza_positivity", secondary_y=True,
         ax=ax, color="gray", legend=True);

### ILI outbreaks, hospital admissions and severe cases

In [ ]:
hk.get_ili_outbreaks(year=2024).tail(3)

In [ ]:
adm = hk.get_hospital_admissions(year=2024)
adm.plot(x="week_start", y=["admission_rate_0_5_per_10000", "admission_rate_65_plus_per_10000",
                            "admission_rate_all_ages_per_10000"],
         figsize=(11, 4), title="Influenza admission rates per 10,000 — Hong Kong, 2024");

In [ ]:
hk.get_severe_cases(year=2024).tail(3)

## 3. COVID-19 daily situation (archived)

The CHP daily situation reports form a cumulative time series starting 8 January 2020.
Note that the upstream files stopped being updated on **19 March 2023** — the accessor
flags every row accordingly.

In [ ]:
covid = hk.get_covid_situation()
print(covid["date"].min(), "→", covid["date"].max(), f"({len(covid)} days)")
covid.tail(3)

In [ ]:
covid.plot(x="date", y=["confirmed_cases_cumulative", "deaths_cumulative"],
           figsize=(11, 4), title="Cumulative COVID-19 cases & deaths — Hong Kong (archived)");